> **About this notebook**
>
> This notebook is part of the training material of the [INDoS COST Action (CA24161)](https://github.com/indos-costaction/) — *Improving Neuroimaging Data for Sharing* — and is available in the repository [indos-costaction/wg3-ts2026-aleman-gomez](https://github.com/indos-costaction/wg3-ts2026-aleman-gomez).
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, CHUV – Lausanne University Hospital)

# FreeSurfer Hands-On: Single-Subject Processing
## Practical

This notebook is a step-by-step, notebook-adapted version of the FreeSurfer single-subject
processing practical. It aims to verify the tools, explore the software's viewer, 
run the cortical reconstruction pipeline on one subject (as a whole and stage-by-stage), inspect the outputs, and learn how to
recognize and fix the most common reconstruction errors.

Throughout, code cells use the Jupyter `!` shell-escape to call FreeSurfer command-line
tools (`recon-all`, `freeview`, ...). A few short Python cells are used instead of plain
shell commands wherever the working directory or an environment variable needs to
**persist across cells** — see the note in Section 1.1 for why that matters.

### Visualizing FreeSurfer results in Neurodesk

GUI tools (`freeview`, `mrview`) need a display. In Neurodesk, that display is the **Neurodesk desktop** (a VNC session). A plain JupyterLab terminal has no display by default.

---

## 1. Start the desktop

- In JupyterLab, click the **NeuroDesktop VNC** launcher tile. Keep that tab open.
- Easiest path: open a terminal **inside the desktop window** and run everything from there.

## 2. Check / set the display

```bash
echo $DISPLAY              # should print something like :1
```

If it's empty (e.g. you are in a Jupyter terminal), point it at the running desktop:

```bash
ls /tmp/.X11-unix/         # e.g. X1  → display :1
export DISPLAY=:1          # use the number you saw
export XAUTHORITY=~/.Xauthority   # only if you get an authorization error
```

## 3. Force software rendering (avoids OpenGL crashes / black windows)

The Neurodesk wrappers run containers with `--cleanenv`, so variables must use the container prefix to reach the app:

```bash
export SINGULARITYENV_LIBGL_ALWAYS_SOFTWARE=1
export APPTAINERENV_LIBGL_ALWAYS_SOFTWARE=1
```

> Tip: add the lines from steps 2–3 to `~/.bashrc` so they're set in every new terminal.

## 4. FreeSurfer → Freeview

```bash
ml av freesurfer               # list versions. It is not mandatory
ml freesurfer/7.4.1
```

Note: `freeview` itself runs without a license; `recon-all` and other processing tools need `export FS_LICENSE=/path/to/license.txt`.

## 5. Where your data must live

Files must be on a path mounted into Neurodesk — on a local install typically `~/neurodesktop-storage` (seen as `/neurodesktop-storage` inside).

---

## Troubleshooting

| Error | Cause | Fix |
|---|---|---|
| `could not connect to display` / `no Qt platform plugin could be initialized` | `$DISPLAY` empty or wrong | Launch from the desktop terminal, or `export DISPLAY=:N` (step 2) |
| `Authorization required` / `No protocol specified` | X auth not passed | `export XAUTHORITY=~/.Xauthority` |
| Black window, GL/GLX errors, segfault on open | No hardware OpenGL in VNC | `export SINGULARITYENV_LIBGL_ALWAYS_SOFTWARE=1` |
| `/tmp/.X11-unix/` is empty | Desktop not running | Open the Desktop tile first |
| Remote / HPC over SSH | No local X server | Use the Neurodesk desktop in the browser, or `ssh -X` / `ssh -Y` with an X server on your machine |

## Learning objectives

By the end of this practical, you should be able to:

- Verify that FreeSurfer and `freeview` are correctly installed and configured.
- Load volumes, surfaces, color look-up tables, and parcellation annotations in `freeview`.
- Convert a NIfTI T1-weighted image into a new FreeSurfer subject.
- Run the full `recon-all` cortical reconstruction pipeline, or run it stage-by-stage
  (`-autorecon1`, `-autorecon2`, `-autorecon3`).
- Compute the Local Gyrification Index (LGI).

<a id="tutorial-outline"></a>
## Tutorial Outline

This tutorial covers the FreeSurfer cortical reconstruction of a single subject, from the BIDS T1-weighted image to cortical parcellations and morphometric statistics.

* **🔷 D. Setting up the data-set**
  * 🔸 D.1. [**Section 0: Downloading the dataset**](#sec0-dataset)

* **🔷 F. FreeSurfer Hands-ON**
  * 🔸 F.1. [**Section 1: Verifying FreeSurfer Installation**](#sec1-install)
  * 🔸 F.2. [**Section 2: Visualizing images with Freeview**](#sec2-freeview)
  * 🔸 F.3. [**Section 3: Converting from NIfTI**](#sec3-convert)
  * 🔸 F.4. [**Section 4: Running recon-all (autorecon1)**](#sec4-autorecon1)
  * 🔸 F.5. [**Section 5: Running recon-all (autorecon2)**](#sec5-autorecon2)
  * 🔸 F.6. [**Section 6: Running recon-all (autorecon3)**](#sec6-autorecon3)
  * 🔸 F.7. [**Section 7: Local Gyrification Index**](#sec7-lgi)
  * 🔸 F.8. [**Section 8: Creating extra parcellations**](#sec8-extra)

---

<a id="sec0-dataset"></a>
## Section 0: Downloading the dataset

All the steps listed in this section are common to most analyses, whether they are studies of one or a group of subjects.

**Section Contents:**
* 🔸 0.1. [**Downloading the Dataset**](#dataset-download): Downloading the dataset
* 🔸 0.2. [**Importing Modules**](#dataset-import): Importing the mandatory modules

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="dataset-download"></a>
* 🔸 0.1. **`Dataset`**: Downloading the dataset

[↑ Back to Section 0 Outline](#sec0-dataset)

---

In [1]:
import os
import re
import tarfile
import zipfile
import urllib.request

# ===================================================================
# --- Download, extract and clean up ---
# ===================================================================
url = "https://filesender.switch.ch/filesender2/download.php?token=6950473c-8a2c-4be3-8df0-69b705830423&files_ids=1361960"
dest_dir = os.path.expanduser("~/neurodesktop-storage")
os.makedirs(dest_dir, exist_ok=True)

aparc_file = os.path.join(
    dest_dir, "0-INDOs-T1w_and_DWI_HandsON-Data", "BIDS", "derivatives_solved",
    "freesurfer", "sub-0001_ses-01_acq-mprage_run-01", "mri", "aparc+aseg.mgz"
)

if not os.path.isfile(aparc_file):
    print(f"Downloading {url}")
    with urllib.request.urlopen(url) as resp:
        ctype = resp.headers.get("Content-Type", "")
        if "text/html" in ctype:
            raise RuntimeError("Got a web page, not a file. Use the direct download link "
                               "(right-click the Download button → Copy link address).")

        # Real file name from the server, if provided
        cd = resp.headers.get("Content-Disposition", "")
        m = re.search(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?', cd)
        fname = m.group(1) if m else "download.tar.gz"
        file_path = os.path.join(dest_dir, fname)

        with open(file_path, "wb") as f:
            while chunk := resp.read(1024 * 1024):
                f.write(chunk)

    print(f"Saved {file_path} ({os.path.getsize(file_path) / 1e6:.1f} MB)")

    # Extract depending on the archive type
    if tarfile.is_tarfile(file_path):
        with tarfile.open(file_path, "r:*") as tf:   # auto-detects .gz/.bz2/.xz
            n = len(tf.getmembers())
            try:
                tf.extractall(dest_dir, filter="data")   # safe extraction (Python ≥3.12 / patched 3.8+)
            except TypeError:
                tf.extractall(dest_dir)                  # older Python without the filter argument
        print(f"Extracted {n} entries to {dest_dir}")
        os.remove(file_path)
        print(f"Removed {file_path}")
    elif zipfile.is_zipfile(file_path):
        with zipfile.ZipFile(file_path) as zf:
            zf.extractall(dest_dir)
            print(f"Extracted {len(zf.namelist())} files to {dest_dir}")
        os.remove(file_path)
        print(f"Removed {file_path}")
    else:
        print("Not a recognised archive (tar/zip) — kept as is.")
else:
    print(f"Data already present: {aparc_file}")

Data already present: /home/jovyan/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/aparc+aseg.mgz


<a id="dataset-import"></a>
* 🔸 0.2. **`Import`**: Importing mandatory modules

[↑ Back to Section 0 Outline](#sec0-dataset)

---

In [2]:
import module
await module.load('freesurfer/7.4.1')
await module.list()

['freesurfer/7.4.1']

<a id="sec1-install"></a>
## Section 1: Verifying the FreeSurfer Installation

In this section we define the working directories, configure the FreeSurfer license and the display, and verify that FreeSurfer runs correctly.

**Section Contents:**
* 🔸 1.1. [**Directories and Variables**](#fs-vars): Defining the working directories
* 🔸 1.2. [**License and Environment**](#fs-license): Passing the license and SUBJECTS_DIR to the container
* 🔸 1.3. [**Display**](#fs-display): Connecting freeview to the Neurodesk desktop
* 🔸 1.4. [**Installation Check**](#fs-check): Verifying the FreeSurfer tools

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="fs-vars"></a>
* 🔸 1.1. **`Directories and Variables`**: Defining the working directories

FreeSurfer locates (and creates) subject data through the `SUBJECTS_DIR` environment
variable. In this tutorial it will point to `$freesurferDir` (your working copy), while the
already processed subject lives in `$freesurferDir_proc` (for comparison, and as a fallback
for the steps that take too long to run during the session).

**A Jupyter-specific gotcha:** a shell assignment typed as `!SUBJECTS_DIR=...` (or even
`!export SUBJECTS_DIR=...`) runs inside a throwaway subshell and is discarded the moment
the cell finishes — it will **not** be visible to `recon-all`/`freeview` in later cells,
and it will not survive a plain `!cd`, either (same issue as `!cd` not persisting the
working directory). The reliable way to set an environment variable and a working
directory that *do* persist for the rest of the notebook is to set them from Python
(`os.environ[...]` and `os.chdir(...)`), since `!` commands are spawned as children of
the notebook's own Python process and inherit its environment and current directory.

[↑ Back to Section 1 Outline](#sec1-install)

---

In [3]:
import os

# Base directory
os.environ['baseDir'] = '/data/neurodesktop-storage'
# bash: export baseDir=/home/dcourse/Data

# For python
baseDir = os.environ['baseDir']

subj_id = '0001'
session_id = '01'
run_id = '01'

# Root Tutorial Directory
os.environ['tutorialDir'] = os.environ['baseDir'] 
# bash: export tutorialDir=$baseDir/Data/02-FibTracking
tutorialDir = os.environ['tutorialDir']

# Directory containing the Nifti files in BIDS format
os.environ['bidsDir'] = os.environ['tutorialDir'] + '/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS'
# bash: export bidsDir=$tutorialDir/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS
bidsDir = os.environ['bidsDir']

# Directory containing the T1w Nifti files in BIDS format
os.environ['t1wDir'] = os.environ['bidsDir'] + '/sub-' + subj_id + '/ses-'+ session_id + '/anat'
# bash: export t1wDir=$bidsDir/sub-0001/ses-01/anat
t1wDir = os.environ['t1wDir']

# Output directory: derivatives (your working directory)
os.environ['derivDir'] = os.environ['bidsDir'] + '/derivatives'
# bash: export derivDir=$bidsDir/derivatives
derivDir = os.environ['derivDir']

# Reference directory: contains pre-processed/solved data for comparison
os.environ['derivDir_proc'] = os.environ['bidsDir'] + '/derivatives_solved'
# bash: export derivDir_proc=$bidsDir/derivatives_solved
derivDir_proc = os.environ['derivDir_proc']

# Output directory for the processing using FreeSurfer
os.environ['freesurferDir'] = os.environ['derivDir'] + '/freesurfer'
# bash: export freesurferDir=$derivDir/freesurfer
freesurferDir = os.environ['freesurferDir'] 

# Reference FreeSurfer directory (already processed subject)
os.environ['freesurferDir_proc'] = os.environ['derivDir_proc'] + '/freesurfer'
# bash: export freesurferDir_proc=$derivDir_proc/freesurfer
freesurferDir_proc = os.environ['freesurferDir_proc']

# FreeSurfer subject ID and input T1w image. 
os.environ['fsSubj'] = f"sub-{subj_id}_ses-{session_id}_acq-mprage_run-{run_id}"
# bash: export fsSubj=sub-0001_ses-01_acq-mprage_run-01
fsSubj = os.environ['fsSubj']

os.environ['t1wFile'] = os.environ['t1wDir'] + '/' + fsSubj + '_T1w.nii.gz'
# bash: export t1wFile=$t1wDir/${fsSubj}_T1w.nii.gz
t1wFile = os.environ['t1wFile']

# Output directory for the parcellation files
os.environ['parcDir'] = os.environ['derivDir'] + '/chimera'
# bash: export parcDir=$derivDir/parcellation
parcDir = os.environ['parcDir']

# Temporal directory for intermediate steps
os.environ['tmpDir'] = os.environ['tutorialDir'] + '/0-INDOs-T1w_and_DWI_HandsON-Data/tmp'
# bash: export tmpDir=$tutorialDir/0-INDOs-T1w_and_DWI_HandsON-Data/tmp
tmpDir = os.environ['tmpDir']

# Create directories
!mkdir -p $tmpDir $freesurferDir $parcDir
# bash: mkdir -p $tmpDir $freesurferDir $parcDir

# Verify the paths
print(f"Tutorial Directory: {os.environ['tutorialDir']}")
print(f"FreeSurfer Directory: {os.environ['freesurferDir']}")
print(f"Temporary Directory: {os.environ['tmpDir']}")
print(f"FreeSurfer subject: {fsSubj}")
print(f"T1w image: {t1wFile}  (exists: {os.path.isfile(t1wFile)})")

Tutorial Directory: /data/neurodesktop-storage
FreeSurfer Directory: /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer
Temporary Directory: /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/tmp
FreeSurfer subject: sub-0001_ses-01_acq-mprage_run-01
T1w image: /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/sub-0001/ses-01/anat/sub-0001_ses-01_acq-mprage_run-01_T1w.nii.gz  (exists: True)


<a id="fs-license"></a>
* 🔸 1.2. **`License and Environment`**: Passing the license and SUBJECTS_DIR to the container

**Why this is needed**

In Neurodesk every FreeSurfer command runs inside a container started with `--cleanenv`,
which drops the notebook's environment variables. A variable only reaches the container if it
is also defined with the `SINGULARITYENV_` / `APPTAINERENV_` prefix. The helper below sets
the three versions at once.

`recon-all` (and most processing tools) need a valid **FreeSurfer license**
(free registration at https://surfer.nmr.mgh.harvard.edu/registration.html). Copy your
`license.txt` into the storage folder so that it is visible inside the container.
`freeview` runs without a license.

[↑ Back to Section 1 Outline](#sec1-install)

---

In [4]:
# ===================================================================
# --- FreeSurfer License and Environment ---
# ===================================================================
print("\n" + "="*70)
print("--- FreeSurfer License and Environment ---")
print("="*70 + "\n")

def set_container_env(name, value):
    """Set a variable for the notebook and for the Neurodesk containers."""
    for prefix in ("", "SINGULARITYENV_", "APPTAINERENV_"):
        os.environ[prefix + name] = str(value)

# License file (edit this path if needed)
fsLicense = f"{baseDir}/license.txt"
set_container_env("FS_LICENSE", fsLicense)
print(f"FS_LICENSE: {fsLicense}  (exists: {os.path.isfile(fsLicense)})")

# Subjects directory
set_container_env("SUBJECTS_DIR", freesurferDir)
print(f"SUBJECTS_DIR: {freesurferDir}")

# Note: we also pass -sd $SUBJECTS_DIR explicitly to recon-all below, so it
# works even if the variable does not reach the container.


--- FreeSurfer License and Environment ---

FS_LICENSE: /data/neurodesktop-storage/license.txt  (exists: True)
SUBJECTS_DIR: /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer


<a id="fs-display"></a>
* 🔸 1.3. **`Display`**: Connecting freeview to the Neurodesk desktop

`freeview` needs a display. Open the **Desktop** tile in JupyterLab (keep that tab open);
the cell below points the notebook to it and forces software rendering, which avoids
black windows and OpenGL crashes inside the VNC session.

> While a `freeview` window is open the cell stays busy. **Close the window to continue.**

[↑ Back to Section 1 Outline](#sec1-install)

---

In [5]:
# ===================================================================
# --- Display Configuration ---
# ===================================================================
print("\n" + "="*70)
print("--- Display Configuration ---")
print("="*70 + "\n")

xdisplays = sorted(os.listdir("/tmp/.X11-unix")) if os.path.isdir("/tmp/.X11-unix") else []

if not os.environ.get("DISPLAY"):
    if xdisplays:
        os.environ["DISPLAY"] = ":" + xdisplays[0].lstrip("X")
    else:
        print("⚠️  No display found. Open the Desktop tile first and re-run this cell.")

set_container_env("LIBGL_ALWAYS_SOFTWARE", 1)
print(f"DISPLAY: {os.environ.get('DISPLAY', '(not set)')}")


--- Display Configuration ---

DISPLAY: :1


<a id="fs-check"></a>
* 🔸 1.4. **`Installation Check`**: Verifying the FreeSurfer tools

[↑ Back to Section 1 Outline](#sec1-install)

---

In [6]:
# ===================================================================
# --- Checking the FreeSurfer Installation ---
# ===================================================================
print("\n" + "="*70)
print("--- Checking the FreeSurfer Installation ---")
print("="*70 + "\n")

!recon-all -version
# bash: recon-all -version

!which recon-all freeview mri_convert
# bash: which recon-all freeview mri_convert


--- Checking the FreeSurfer Installation ---

freesurfer-linux-centos8_x86_64-7.4.1-20230613-7eb8460
/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_7.4.1_20231214/recon-all
/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_7.4.1_20231214/freeview
/cvmfs/neurodesk.ardc.edu.au/containers/freesurfer_7.4.1_20231214/mri_convert


<a id="sec2-freeview"></a>
## Section 2: Visualizing images with Freeview

Before processing our own subject, we explore `freeview` using the already processed subject in `$freesurferDir_proc`.

**Section Contents:**
* 🔸 2.1. [**Volumes**](#fv-volumes): Loading volumes and a color look-up table
* 🔸 2.2. [**Surfaces**](#fv-surfaces): Overlaying the white and pial surfaces
* 🔸 2.3. [**Annotations**](#fv-annot): Displaying a cortical parcellation on the inflated surface

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="fv-volumes"></a>
* 🔸 2.1. **`Volumes`**: Loading volumes and a color look-up table

**Useful `freeview` volume options** (appended to the file name with `:`):

- `colormap=grayscale|heat|jet|lut` — `lut` uses FreeSurfer's color table (`FreeSurferColorLUT.txt`)
- `opacity=0–1`
- `visible=0` to load a layer hidden

Keyboard shortcuts in the viewer: `Page Up/Down` or arrow keys to move through slices,
`Ctrl+1/2/3` to switch sagittal/coronal/axial views, `Alt+c` to cycle through color maps.

[↑ Back to Section 2 Outline](#sec2-freeview)

---

In [7]:
# ===================================================================
# --- Visualizing Volumes ---
# ===================================================================
print("\n" + "="*70)
print("--- Visualizing Volumes ---")
print("="*70 + "\n")

refSubjDir = f"{freesurferDir_proc}/{fsSubj}"

!echo freeview -v {refSubjDir}/mri/T1.mgz \
             {refSubjDir}/mri/brainmask.mgz \
             {refSubjDir}/mri/aseg.mgz:colormap=lut:opacity=0.3
# bash: freeview -v $freesurferDir_proc/$fsSubj/mri/T1.mgz $freesurferDir_proc/$fsSubj/mri/brainmask.mgz $freesurferDir_proc/$fsSubj/mri/aseg.mgz:colormap=lut:opacity=0.3


--- Visualizing Volumes ---

freeview -v /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/T1.mgz /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/brainmask.mgz /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/aseg.mgz:colormap=lut:opacity=0.3


<a id="fv-surfaces"></a>
* 🔸 2.2. **`Surfaces`**: Overlaying the white and pial surfaces

- **White surface** (`?h.white`): boundary between white matter and grey matter.
- **Pial surface** (`?h.pial`): boundary between grey matter and CSF.

Surface options: `edgecolor=`, `color=`, `overlay=`, `annot=`.

[↑ Back to Section 2 Outline](#sec2-freeview)

---

In [8]:
# ===================================================================
# --- Visualizing Surfaces ---
# ===================================================================
print("\n" + "="*70)
print("--- Visualizing Surfaces ---")
print("="*70 + "\n")

!echo freeview -v {refSubjDir}/mri/brainmask.mgz \
          -f {refSubjDir}/surf/lh.white:edgecolor=blue \
             {refSubjDir}/surf/lh.pial:edgecolor=red \
             {refSubjDir}/surf/rh.white:edgecolor=blue \
             {refSubjDir}/surf/rh.pial:edgecolor=red


--- Visualizing Surfaces ---

freeview -v /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/brainmask.mgz -f /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.white:edgecolor=blue /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.pial:edgecolor=red /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/rh.white:edgecolor=blue /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/rh.pial:edgecolor=red


In [30]:
# ===================================================================
# --- Visualizing Surfaces for fsaverage ---
# ===================================================================
print("\n" + "="*70)
print("--- Visualizing Surfaces for fsaverage ---")
print("="*70 + "\n")

ref_fsaverage = f"{freesurferDir_proc}/fsaverage"

!echo freeview -f {ref_fsaverage}/surf/lh.white:edgecolor=blue \
             {ref_fsaverage}/surf/lh.pial:edgecolor=red \
             {ref_fsaverage}/surf/rh.white:edgecolor=blue \
             {ref_fsaverage}/surf/rh.pial:edgecolor=red
    



--- Visualizing Surfaces for fsaverage ---

freeview -f /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/surf/lh.white:edgecolor=blue /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/surf/lh.pial:edgecolor=red /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/surf/rh.white:edgecolor=blue /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/surf/rh.pial:edgecolor=red


<a id="fv-annot"></a>
* 🔸 2.3. **`Annotations`**: Displaying a cortical parcellation on the inflated surface

FreeSurfer produces three cortical parcellations by default:

| Annotation | Atlas | Regions per hemisphere |
|---|---|---|
| `?h.aparc.annot` | Desikan-Killiany | 34 |
| `?h.aparc.a2009s.annot` | Destrieux | 74 |
| `?h.aparc.DKTatlas.annot` | DKT | 31 |

[↑ Back to Section 2 Outline](#sec2-freeview)

---

In [9]:
# ===================================================================
# --- Visualizing Annotations and Cortical Thickness Map ---
# ===================================================================
print("\n" + "="*70)
print("--- Visualizing Annotations ---")
print("="*70 + "\n")

!echo freeview -f {refSubjDir}/surf/lh.inflated:annot={refSubjDir}/label/lh.aparc.annot \
             {refSubjDir}/surf/lh.inflated:overlay={refSubjDir}/surf/lh.thickness:overlay_threshold=1,4:visible=0 \
          -viewport 3d


--- Visualizing Annotations ---

freeview -f /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.inflated:annot=/data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/label/lh.aparc.annot /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.inflated:overlay=/data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.thickness:overlay_threshold=1,4:visible=0 -viewport 3d


In [31]:
# ===================================================================
# --- Visualizing Annotations and Cortical Thickness Map for fsaverage ---
# ===================================================================
print("\n" + "="*70)
print("--- Visualizing Annotations for fsaverage ---")
print("="*70 + "\n")

ref_fsaverage = f"{freesurferDir_proc}/fsaverage"

!echo freeview -f {ref_fsaverage}/surf/lh.inflated:annot={ref_fsaverage}/label/lh.aparc.annot \
             {ref_fsaverage}/surf/lh.inflated:overlay={ref_fsaverage}/surf/lh.thickness:overlay_threshold=1,4:visible=0 \
          -viewport 3d
    



--- Visualizing Annotations for fsaverage ---

freeview -f /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/surf/lh.inflated:annot=/data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/label/lh.aparc.annot /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/surf/lh.inflated:overlay=/data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/fsaverage/surf/lh.thickness:overlay_threshold=1,4:visible=0 -viewport 3d


<a id="sec3-convert"></a>
## Section 3: Converting from NIfTI

`recon-all` works on its own subject folder structure. The first step is to import the T1-weighted NIfTI image into a new FreeSurfer subject.

**Section Contents:**
* 🔸 3.1. [**Inspecting the T1w**](#convert-inspect): Checking the header of the input image
* 🔸 3.2. [**Creating the Subject**](#convert-create): Creating the FreeSurfer subject folder

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="convert-inspect"></a>
* 🔸 3.1. **`Inspecting the T1w`**: Checking the header of the input image

Check voxel size, dimensions and orientation. FreeSurfer expects roughly 1 mm isotropic
T1-weighted images; for sub-millimetre data see the `-hires` flag.

[↑ Back to Section 3 Outline](#sec3-convert)

---

In [10]:
# ===================================================================
# --- Inspecting the T1w Image ---
# ===================================================================
print("\n" + "="*70)
print("--- Inspecting the T1w Image ---")
print("="*70 + "\n")

!mri_info {t1wFile}
# bash: mri_info $t1wFile


--- Inspecting the T1w Image ---

Volume information for /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/sub-0001/ses-01/anat/sub-0001_ses-01_acq-mprage_run-01_T1w.nii.gz
          type: nii
    dimensions: 160 x 256 x 256
   voxel sizes: 1.000000, 0.976562, 0.976562
          type: SHORT (4)
           fov: 160.000
           dof: 1
        xstart: -80.0, xend: 80.0
        ystart: -125.0, yend: 125.0
        zstart: -125.0, zend: 125.0
            TR: 1910.00 msec, TE: 0.00 msec, TI: 0.00 msec, flip angle: 0.00 degrees
       nframes: 1
       PhEncDir: UNKNOWN
       FieldStrength: 0.000000
ras xform present
    xform info: x_r =   0.9978, y_r =   0.0584, z_r =   0.0301, c_r =    -1.5360
              : x_a =  -0.0588, y_a =   0.9982, z_a =   0.0140, c_a =    13.2384
              : x_s =  -0.0292, y_s =  -0.0157, z_s =   0.9994, c_s =   -10.4495
Orientation   : RAS
Primary Slice Direction: axial

voxel to ras transform:
                0.9978   0.0570   0.0294   -

<a id="convert-create"></a>
* 🔸 3.2. **`Creating the Subject`**: Creating the FreeSurfer subject folder

`recon-all -i` converts the image to `mri/orig/001.mgz` and creates the folder structure
(`mri`, `surf`, `label`, `stats`, `scripts`, ...). With several T1w acquisitions, pass one
`-i` per image and they will be averaged in the motion-correction step.

[↑ Back to Section 3 Outline](#sec3-convert)

---

In [12]:
# ===================================================================
# --- Creating the FreeSurfer Subject ---
# ===================================================================
print("\n" + "="*70)
print("--- Creating the FreeSurfer Subject ---")
print("="*70 + "\n")

!recon-all -i {t1wFile} -s {fsSubj} -sd {freesurferDir}
# bash: recon-all -i $t1wFile -s $fsSubj -sd $freesurferDir

!ls {freesurferDir}/{fsSubj}


--- Creating the FreeSurfer Subject ---

fs-check-version --s sub-0001_ses-01_acq-mprage_run-01 --o /tmp/tmp.c8NvV1
Mon Sep 28 16:18:23 UTC 2026

setenv SUBJECTS_DIR /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer
cd /data/Docencia/INDOs-2026/wg3-ts2026-aleman-gomez/HandsOn
/opt/freesurfer-7.4.1/bin/fs-check-version --s sub-0001_ses-01_acq-mprage_run-01 --o /tmp/tmp.c8NvV1
-rwxrwxr-x 1 nobody nobody 18565 Jun 14  2023 /opt/freesurfer-7.4.1/bin/fs-check-version

freesurfer-linux-centos8_x86_64-7.4.1-20230613-7eb8460
$Id$
Linux 3ef9c5be7e6e 6.14.0-37-generic #37-Ubuntu SMP PREEMPT_DYNAMIC Fri Nov 14 22:10:32 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux
pid 37385
Current FS Version freesurfer-linux-centos8_x86_64-7.4.1-20230613-7eb8460
Subject does not have a bstampfile, copying /opt/freesurfer-7.4.1/build-stamp.txt
Subject FS Version: freesurfer-linux-centos8_x86_64-7.4.1-20230613-7eb8460
No constraints on version because REQ=UnSet and FsVerFile=NotThe

<a id="sec4-autorecon1"></a>
## Section 4: Running recon-all (autorecon1)

`-autorecon1` performs motion correction, intensity normalization, Talairach registration and skull stripping.

**Section Contents:**
* 🔸 4.1. [**Running autorecon1**](#ar1-run): Motion correction to skull stripping
* 🔸 4.2. [**Quality Control and Fixing Errors**](#ar1-qc): Checking the skull stripping and correcting a bad skull strip

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="ar1-run"></a>
* 🔸 4.1. **`Running autorecon1`**: Motion correction to skull stripping

**Main outputs** (in `mri/`): `orig.mgz`, `nu.mgz` (bias-corrected), `T1.mgz` (intensity
normalized), `transforms/talairach.xfm`, `brainmask.mgz` (skull stripped).

Expected run time: around 15–30 minutes.

[↑ Back to Section 4 Outline](#sec4-autorecon1)

---

In [13]:
# ===================================================================
# --- Running autorecon1 ---
# ===================================================================
print("\n" + "="*70)
print("--- Running autorecon1 ---")
print("="*70 + "\n")

!echo recon-all -s {fsSubj} -sd {freesurferDir} -autorecon1 -threads 4
# bash: recon-all -s $fsSubj -sd $freesurferDir -autorecon1 -threads 4


--- Running autorecon1 ---

recon-all -s sub-0001_ses-01_acq-mprage_run-01 -sd /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer -autorecon1 -threads 4


<a id="ar1-qc"></a>
* 🔸 4.2. **`Quality Control and Fixing Errors`**: Checking the skull stripping and correcting a bad skull strip

Look for **remaining skull/dura** (too little removed) or **missing brain**, typically in
the superior or temporal cortex (too much removed). If your own run is not finished, the
cell falls back to the processed subject.

[↑ Back to Section 4 Outline](#sec4-autorecon1)

---

In [14]:
# ===================================================================
# --- Checking the Skull Stripping ---
# ===================================================================
print("\n" + "="*70)
print("--- Checking the Skull Stripping ---")
print("="*70 + "\n")

subjDir = f"{freesurferDir}/{fsSubj}"
if not os.path.isfile(f"{subjDir}/mri/brainmask.mgz"):
    print("Using the already processed subject")
    subjDir = f"{freesurferDir_proc}/{fsSubj}"

!echo freeview -v {subjDir}/mri/T1.mgz \
             {subjDir}/mri/brainmask.mgz:colormap=heat:opacity=0.4


--- Checking the Skull Stripping ---

Using the already processed subject
freeview -v /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/T1.mgz /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/brainmask.mgz:colormap=heat:opacity=0.4


<a id="sec5-autorecon2"></a>
## Section 5: Running recon-all (autorecon2)

`-autorecon2` performs the subcortical segmentation, white matter segmentation and the creation of the white and pial surfaces. It is the longest part of the pipeline, so during the session we inspect the already processed subject.

**Section Contents:**
* 🔸 5.1. [**Running autorecon2**](#ar2-run): Subcortical segmentation and surface creation
* 🔸 5.2. [**Quality Control and Fixing Errors**](#ar2-qc): Checking segmentation and surfaces and control points and white matter edits

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="ar2-run"></a>
* 🔸 5.1. **`Running autorecon2`**: Subcortical segmentation and surface creation ⚠️ *(Do not run)*

**Main outputs:** `mri/aseg.presurf.mgz`, `mri/wm.mgz`, `mri/filled.mgz`, `surf/?h.orig`,
`surf/?h.white`, `surf/?h.pial`, `surf/?h.inflated`, `surf/?h.sphere`.

Expected run time: several hours.

[↑ Back to Section 5 Outline](#sec5-autorecon2)

---

In [15]:
# ===================================================================
# --- Running autorecon2 ---
# ===================================================================
print("\n" + "="*70)
print("--- Running autorecon2 ---")
print("="*70 + "\n")

!echo recon-all -s {fsSubj} -sd {freesurferDir} -autorecon2 -threads 4
# bash: recon-all -s $fsSubj -sd $freesurferDir -autorecon2 -threads 4


--- Running autorecon2 ---

recon-all -s sub-0001_ses-01_acq-mprage_run-01 -sd /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer -autorecon2 -threads 4


<a id="ar2-qc"></a>
* 🔸 5.2. **`Quality Control and Fixing Errors`**: Checking segmentation and surfaces and control points and white matter edits

Things to check, slice by slice:

- The **white surface** follows the WM/GM boundary (not cutting into WM, not leaking into GM).
- The **pial surface** does not include dura, skull or cerebellum.
- `wm.mgz` has no holes or missing gyri (common in the temporal poles and orbitofrontal cortex).
- `aseg.mgz` labels fit their structures.

[↑ Back to Section 5 Outline](#sec5-autorecon2)

---

In [16]:
# ===================================================================
# --- Checking Segmentation and Surfaces ---
# ===================================================================
print("\n" + "="*70)
print("--- Checking Segmentation and Surfaces ---")
print("="*70 + "\n")

subjDir = f"{freesurferDir}/{fsSubj}"
if not os.path.isfile(f"{subjDir}/surf/lh.pial"):
    print("Using the already processed subject")
    subjDir = f"{freesurferDir_proc}/{fsSubj}"

!echo freeview -v {subjDir}/mri/brainmask.mgz \
             {subjDir}/mri/wm.mgz:colormap=heat:opacity=0.4:visible=0 \
             {subjDir}/mri/aseg.mgz:colormap=lut:opacity=0.2:visible=0 \
          -f {subjDir}/surf/lh.white:edgecolor=blue \
             {subjDir}/surf/lh.pial:edgecolor=red \
             {subjDir}/surf/rh.white:edgecolor=blue \
             {subjDir}/surf/rh.pial:edgecolor=red


--- Checking Segmentation and Surfaces ---

Using the already processed subject
freeview -v /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/brainmask.mgz /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/wm.mgz:colormap=heat:opacity=0.4:visible=0 /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/aseg.mgz:colormap=lut:opacity=0.2:visible=0 -f /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.white:edgecolor=blue /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.pial:edgecolor=red /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesu


| Error | Edit | Re-run |
|---|---|---|
| White surface misses WM due to low intensity | Add control points (`tmp/control.dat`) | `-autorecon2-cp -autorecon3` |
| Holes or missing WM in `wm.mgz` | Fill voxels in `wm.mgz` (value 255) | `-autorecon2-wm -autorecon3` |
| Pial includes dura / non-brain | Erase voxels in `brainmask.mgz` | `-autorecon-pial` |

[↑ Back to Section 5 Outline](#sec5-autorecon2)

---

In [19]:
# ===================================================================
# --- Re-running after Manual Edits ---
# ===================================================================
print("\n" + "="*70)
print("--- Re-running after Manual Edits ---")
print("="*70 + "\n")

# After control points
# !recon-all -s {fsSubj} -sd {freesurferDir} -autorecon2-cp -autorecon3

# After white matter edits
# !recon-all -s {fsSubj} -sd {freesurferDir} -autorecon2-wm -autorecon3

# After brainmask edits (pial surface)
# !recon-all -s {fsSubj} -sd {freesurferDir} -autorecon-pial


--- Re-running after Manual Edits ---



<a id="sec6-autorecon3"></a>
## Section 6: Running recon-all (autorecon3)

`-autorecon3` registers the surfaces to the spherical atlas, creates the cortical parcellations and computes the morphometric statistics.

**Section Contents:**
* 🔸 6.1. [**Running autorecon3**](#ar3-run): Spherical registration, parcellation and statistics
* 🔸 6.2. [**Morphometric Statistics**](#ar3-stats): Reading the stats files

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="ar3-run"></a>
* 🔸 6.1. **`Running autorecon3`**: Spherical registration, parcellation and statistics ⚠️ *(Do not run)*

**Main outputs:** `surf/?h.sphere.reg`, `surf/?h.thickness`, `surf/?h.area`, `surf/?h.volume`,
`label/?h.aparc*.annot`, `mri/aparc+aseg.mgz`, `stats/*.stats`.

Expected run time: 1–2 hours.

[↑ Back to Section 6 Outline](#sec6-autorecon3)

---

In [17]:
# ===================================================================
# --- Running autorecon3 ---
# ===================================================================
print("\n" + "="*70)
print("--- Running autorecon3 ---")
print("="*70 + "\n")

!echo recon-all -s {fsSubj} -sd {freesurferDir} -autorecon3 -threads 4
# bash: recon-all -s $fsSubj -sd $freesurferDir -autorecon3 -threads 4

# The complete pipeline can also be run in a single call:
# !recon-all -s {fsSubj} -sd {freesurferDir} -all -threads 4


--- Running autorecon3 ---

recon-all -s sub-0001_ses-01_acq-mprage_run-01 -sd /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer -autorecon3 -threads 4


<a id="ar3-stats"></a>
* 🔸 6.2. **`Morphometric Statistics`**: Reading the stats files

- `stats/aseg.stats`: volumes of subcortical structures, plus global measures (eTIV, total GM, ...).
- `stats/?h.aparc.stats`: surface area, GM volume, mean thickness, curvature... per cortical region.

The small function below reads any FreeSurfer stats table into a `pandas` DataFrame.

[↑ Back to Section 6 Outline](#sec6-autorecon3)

---

In [18]:
# ===================================================================
# --- Reading the Morphometric Statistics ---
# ===================================================================
print("\n" + "="*70)
print("--- Reading the Morphometric Statistics ---")
print("="*70 + "\n")

import pandas as pd

def read_fs_stats(stats_file):
    """Read the table of a FreeSurfer .stats file into a DataFrame."""
    with open(stats_file) as f:
        lines = f.readlines()
    cols = [l.split()[2:] for l in lines if l.startswith("# ColHeaders")][0]
    rows = [l.split() for l in lines if not l.startswith("#")]
    df = pd.DataFrame(rows, columns=cols)
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
        except ValueError:
            pass
    return df

subjDir = f"{freesurferDir}/{fsSubj}"
if not os.path.isfile(f"{subjDir}/stats/lh.aparc.stats"):
    print("Using the already processed subject")
    subjDir = f"{freesurferDir_proc}/{fsSubj}"

aseg = read_fs_stats(f"{subjDir}/stats/aseg.stats")
lh_aparc = read_fs_stats(f"{subjDir}/stats/lh.aparc.stats")

display(aseg[["StructName", "Volume_mm3"]].head(10))
display(lh_aparc[["StructName", "SurfArea", "GrayVol", "ThickAvg"]].head(10))


--- Reading the Morphometric Statistics ---

Using the already processed subject


,StructName,Volume_mm3
0,Left-Lateral-Ventricle,14023.0
1,Left-Inf-Lat-Vent,462.8
2,Left-Cerebellum-White-Matter,15027.6
3,Left-Cerebellum-Cortex,62089.9
4,Left-Thalamus,7746.9
5,Left-Caudate,3361.6
6,Left-Putamen,4555.9
7,Left-Pallidum,2179.8
8,3rd-Ventricle,1049.5
9,4th-Ventricle,1623.6


,StructName,SurfArea,GrayVol,ThickAvg
0,bankssts,1033,2521,2.609
1,caudalanteriorcingulate,447,1100,2.301
2,caudalmiddlefrontal,1997,5909,2.677
3,cuneus,1370,2673,1.864
4,entorhinal,524,1899,3.022
5,fusiform,3050,7984,2.440
6,inferiorparietal,4805,12930,2.456
7,inferiortemporal,3761,12529,2.809
8,isthmuscingulate,1024,2594,2.247
9,lateraloccipital,4042,8276,1.921


<a id="sec7-lgi"></a>
## Section 7: Local Gyrification Index

The Local Gyrification Index (LGI; Schaer et al., 2008) measures, at each vertex, how much cortex is buried in the sulci within a circular region of the outer hull surface.

**Section Contents:**
* 🔸 7.1. [**Computing the LGI**](#lgi-run): Running recon-all -localGI
* 🔸 7.2. [**Visualizing the LGI**](#lgi-viz): Displaying the LGI on the inflated surface

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="lgi-run"></a>
* 🔸 7.1. **`Computing the LGI`**: Running recon-all -localGI ⚠️ *(Do not run)*

**Note:** `-localGI` relies on **MATLAB** scripts. If MATLAB is not available in the
container, use the already processed output (`surf/?h.pial_lgi`).

Output: `surf/lh.pial_lgi`, `surf/rh.pial_lgi`.

[↑ Back to Section 7 Outline](#sec7-lgi)

---

In [21]:
# ===================================================================
# --- Computing the Local Gyrification Index ---
# ===================================================================
print("\n" + "="*70)
print("--- Computing the Local Gyrification Index ---")
print("="*70 + "\n")
freesurferDir =
!recon-all -s {fsSubj} -sd {freesurferDir} -localGI
# bash: recon-all -s $fsSubj -sd $freesurferDir -localGI


--- Computing the Local Gyrification Index ---

recon-all -s sub-0001_ses-01_acq-mprage_run-01 -sd /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer -localGI


<a id="lgi-viz"></a>
* 🔸 7.2. **`Visualizing the LGI`**: Displaying the LGI on the inflated surface

[↑ Back to Section 7 Outline](#sec7-lgi)

---

In [19]:
# ===================================================================
# --- Visualizing the Local Gyrification Index ---
# ===================================================================
print("\n" + "="*70)
print("--- Visualizing the Local Gyrification Index ---")
print("="*70 + "\n")

subjDir = f"{freesurferDir}/{fsSubj}"
if not os.path.isfile(f"{subjDir}/surf/lh.pial_lgi"):
    print("Using the already processed subject")
    subjDir = f"{freesurferDir_proc}/{fsSubj}"

!echo freeview -f {subjDir}/surf/lh.inflated:overlay={subjDir}/surf/lh.pial_lgi:overlay_threshold=1,5 \
          -viewport 3d


--- Visualizing the Local Gyrification Index ---

Using the already processed subject
freeview -f /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.inflated:overlay=/data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/surf/lh.pial_lgi:overlay_threshold=1,5 -viewport 3d


<a id="sec8-extra"></a>
## Section 8: Creating extra parcellations

Beyond the default atlases, additional parcellations can be derived from the FreeSurfer outputs.

**Section Contents:**
* 🔸 8.1. [**Subcortical Subregions**](#extra-subregions): Thalamic nuclei and hippocampal subfields

[↑ Back to Tutorial Outline](#tutorial-outline)

---

<a id="extra-subregions"></a>
* 🔸 8.1. **`Subcortical Subregions`**: Thalamic nuclei and hippocampal subfields ⚠️ *(Do not run)*

Since FreeSurfer 7.3, `segment_subregions` segments the thalamic nuclei, the hippocampal
subfields and amygdala nuclei, and the brainstem structures. Outputs are written to `mri/`.

[↑ Back to Section 8 Outline](#sec8-extra)

---

In [23]:
# ===================================================================
# --- Segmenting Subcortical Subregions ---
# ===================================================================
print("\n" + "="*70)
print("--- Segmenting Subcortical Subregions ---")
print("="*70 + "\n")

!echo segment_subregions thalamus --cross {fsSubj} --sd {freesurferDir}
!echo segment_subregions hippo-amygdala --cross {fsSubj} --sd {freesurferDir}
!echo segment_subregions brainstem --cross {fsSubj} --sd {freesurferDir}

print("\n" + "="*70)
!echo freeview -v {subjDir}/mri/brainmask.mgz \
             {subjDir}/mri/brainstemSsLabels.mgz:colormap=lut:opacity=0.2:visible=0 \
            {subjDir}/mri/lh.hippoAmygLabels.mgz:colormap=lut:opacity=0.2:visible=0 \
            {subjDir}/mri/lh.hippoAmygLabels.HBT.mgz:colormap=lut:opacity=0.2:visible=0 \
            {subjDir}/mri/ThalamicNuclei.mgz:colormap=lut:opacity=0.2:visible=0 


--- Segmenting Subcortical Subregions ---

segment_subregions thalamus --cross sub-0001_ses-01_acq-mprage_run-01 --sd /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer
segment_subregions hippo-amygdala --cross sub-0001_ses-01_acq-mprage_run-01 --sd /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer
segment_subregions brainstem --cross sub-0001_ses-01_acq-mprage_run-01 --sd /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives/freesurfer

freeview -v /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/brainmask.mgz /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01_acq-mprage_run-01/mri/brainstemSsLabels.mgz:colormap=lut:opacity=0.2:visible=0 /data/neurodesktop-storage/0-INDOs-T1w_and_DWI_HandsON-Data/BIDS/derivatives_solved/freesurfer/sub-0001_ses-01